1. Setup and Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style='whitegrid')
%matplotlib inline

DATA_DIR   = Path('../../dataset')
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
movies  = pd.read_csv(DATA_DIR / 'movies.csv')

print(f'Loaded {len(ratings):,} ratings across {ratings["userId"].nunique()} users and {ratings["movieId"].nunique()} movies.')

# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(ratings['rating'], bins=9, kde=False, color='steelblue', ax=axes[0])
axes[0].set_title('Overall Rating Distribution')
axes[0].set_xlabel('Rating')

ratings_per_user = ratings.groupby('userId').size()
sns.histplot(ratings_per_user, bins=30, kde=True, color='teal', ax=axes[1])
axes[1].set_title('Ratings per User Distribution')
axes[1].set_xlabel('Number of Ratings')
plt.tight_layout()
plt.show()

2. Feature Engineering

In [ ]:
movies['genre_list'] = movies['genres'].apply(lambda g: [x for x in g.split('|') if x != '(no genres listed)'])
all_genres = sorted({g for lst in movies['genre_list'] for g in lst})

# Genre watch share
rated_exp = ratings.merge(movies[['movieId', 'genre_list']], on='movieId').explode('genre_list')
user_genre_cnt  = rated_exp.groupby(['userId', 'genre_list']).size().unstack(fill_value=0)
genre_share_df  = user_genre_cnt.div(user_genre_cnt.sum(axis=1), axis=0).reindex(columns=all_genres, fill_value=0)

# Basic stats
user_stats = ratings.groupby('userId')['rating'].agg(user_mean='mean', user_count='count', user_std='std').fillna(0)

# SVD latent taste
matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
svd = TruncatedSVD(n_components=20, random_state=42)
svd_df = pd.DataFrame(svd.fit_transform(matrix.values), index=matrix.index,
                       columns=[f'svd_{i}' for i in range(20)])

user_ids = sorted(ratings['userId'].unique())
X_df     = user_stats.join(genre_share_df).join(svd_df).reindex(user_ids).fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

print(f'Feature matrix: {X_df.shape[0]} users x {X_df.shape[1]} features')
print(f'  User stats: 3  |  Genre shares: {len(all_genres)}  |  SVD: 20')

3. Visualizing Input Features

In [ ]:
# User stat distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, color in zip(axes, ['user_mean', 'user_count', 'user_std'],
                           ['steelblue', 'orange', 'mediumseagreen']):
    sns.histplot(X_df[col], bins=30, kde=True, color=color, ax=ax)
    ax.set_title(col)
plt.suptitle('User Stat Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Genre watch share
genre_prevalence = genre_share_df.mean().sort_values(ascending=False)
plt.figure(figsize=(14, 4))
sns.barplot(x=genre_prevalence.index, y=genre_prevalence.values, palette='viridis')
plt.title('Average Genre Watch Share Across Users')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Mean Share')
plt.tight_layout()
plt.show()

# SVD explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(svd.explained_variance_ratio_)), svd.explained_variance_ratio_, color='cornflowerblue')
axes[0].set_title('SVD: Variance per Component')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Explained Variance Ratio')

axes[1].plot(np.cumsum(svd.explained_variance_ratio_), color='red', marker='o', markersize=4)
axes[1].set_title('SVD: Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
plt.tight_layout()
plt.show()

print('Raw Input Feature Sample (First 10 Users)')
display(X_df.iloc[:10, :10].style.background_gradient(cmap='coolwarm', axis=0).format('{:.3f}'))

4. K-Means

In [ ]:
print('Searching for optimal K...')
k_results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score  = silhouette_score(X_scaled, labels)
    k_results.append({'K': k, 'Silhouette': score})
    print(f'  K={k}  Silhouette={score:.4f}')

k_df  = pd.DataFrame(k_results)
best_k = int(k_df.loc[k_df['Silhouette'].idxmax(), 'K'])

plt.figure(figsize=(10, 4))
sns.lineplot(data=k_df, x='K', y='Silhouette', marker='o', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K = {best_k}')
plt.title('Optimal Cluster Search — Silhouette Method')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Best K: {best_k}  (Silhouette = {k_df.loc[k_df["K"]==best_k, "Silhouette"].values[0]:.4f})')

# Fit best K-Means and visualize in 2D
best_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = best_km.fit_predict(X_scaled)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels, palette='viridis', alpha=0.7, s=60)
plt.title(f'K-Means User Clusters in 2D (K={best_k})')
plt.xlabel('Principal Component 1 (General Taste)')
plt.ylabel('Principal Component 2 (Niche Preferences)')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

5. Cluster Profiling

In [ ]:
df_viz = X_df.copy()
df_viz['cluster'] = kmeans_labels

cluster_profiles = df_viz.groupby('cluster')[all_genres].mean()
max_val = cluster_profiles.max().max()

# Signature genre bars per cluster
fig, axes = plt.subplots(1, best_k, figsize=(5 * best_k, 4), sharey=False)
if best_k == 1:
    axes = [axes]
colors = sns.color_palette('viridis', best_k)

for cid, ax in enumerate(axes):
    top5 = cluster_profiles.loc[cid].sort_values(ascending=False).head(5)
    ax.barh(top5.index[::-1], top5.values[::-1], color=colors[cid])
    ax.set_title(f'Cluster {cid}\n({(kmeans_labels == cid).sum()} users)')
    ax.set_xlabel('Mean Genre Share')
    ax.set_xlim(0, max_val * 1.1)

plt.suptitle('Signature Genres per Cluster (Top 5)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Mean user stats per cluster
stat_profile = df_viz.groupby('cluster')[['user_mean', 'user_count', 'user_std']].mean()
print('Mean User Stats per Cluster')
display(stat_profile.style.background_gradient(cmap='coolwarm', axis=0).format('{:.3f}'))

6. DBSCAN

In [ ]:
print('Estimating eps with k-NN...')
nn = NearestNeighbors(n_neighbors=5).fit(X_scaled)
dists, _ = nn.kneighbors(X_scaled)
eps_auto  = np.percentile(dists[:, -1], 90)
print(f'Auto eps = {eps_auto:.4f}')

# eps sensitivity plot
sorted_dists = np.sort(dists[:, -1])
plt.figure(figsize=(10, 4))
plt.plot(sorted_dists, color='steelblue')
plt.axhline(eps_auto, color='red', linestyle='--', label=f'eps = {eps_auto:.3f} (90th pct)')
plt.title('k-NN Distance Curve (k=5) — Used to Choose eps')
plt.xlabel('User (sorted)')
plt.ylabel('Distance to 5th Nearest Neighbor')
plt.legend()
plt.tight_layout()
plt.show()

dbscan = DBSCAN(eps=eps_auto, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = list(db_labels).count(-1)
print(f'DBSCAN found {n_clusters} clusters  |  {n_noise} noise users ({n_noise/len(db_labels):.1%})')

plt.figure(figsize=(12, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=db_labels, palette='tab10', alpha=0.6, s=60)
plt.title(f'DBSCAN Clusters (noise = -1)  |  {n_clusters} clusters found')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

7. Algorithm Comparison

In [ ]:
kmeans_sil = silhouette_score(X_scaled, kmeans_labels)

# DBSCAN silhouette only if more than 1 cluster and not all noise
db_mask = db_labels != -1
if len(set(db_labels[db_mask])) > 1:
    dbscan_sil = silhouette_score(X_scaled[db_mask], db_labels[db_mask])
else:
    dbscan_sil = float('nan')

# Cluster size distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

km_sizes = pd.Series(kmeans_labels).value_counts().sort_index()
axes[0].bar(km_sizes.index.astype(str), km_sizes.values, color=sns.color_palette('viridis', best_k))
axes[0].set_title(f'K-Means Cluster Sizes (K={best_k}, Silhouette={kmeans_sil:.4f})')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Users')

db_sizes = pd.Series(db_labels).value_counts().sort_index()
db_colors = ['gray' if c == -1 else 'steelblue' for c in db_sizes.index]
axes[1].bar(db_sizes.index.astype(str), db_sizes.values, color=db_colors)
axes[1].set_title(f'DBSCAN Cluster Sizes (Noise={n_noise}, Silhouette={dbscan_sil:.4f})')
axes[1].set_xlabel('Cluster (-1 = noise)')
axes[1].set_ylabel('Number of Users')
plt.tight_layout()
plt.show()

# Summary table
summary = pd.DataFrame({
    'Algorithm':       ['K-Means', 'DBSCAN'],
    'Type':            ['Centroid-based', 'Density-based'],
    'Clusters Found':  [best_k, n_clusters],
    'Noise Users':     [0, n_noise],
    'Silhouette':      [round(kmeans_sil, 4), round(dbscan_sil, 4) if not np.isnan(dbscan_sil) else 'N/A'],
    'Best For':        ['Defined equal-size groups', 'Arbitrary shapes + outlier detection'],
})
display(summary)
print(f'\nBest cluster quality: K-Means (Silhouette = {kmeans_sil:.4f})')